In [ ]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_metadata_snapshot as ncbi_metadata_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pca_kmeans as pca_kmeans_module
import src.pago_pipeline.pca_kmeans_snapshot as pca_kmeans_snapshot_module
import src.pago_pipeline.sweep_genes_snapshot as sweep_genes_snapshot_module
from src.pago_pipeline.storage import sha256_of_file

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_metadata_snapshot_module = importlib.reload(ncbi_metadata_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pca_kmeans_module = importlib.reload(pca_kmeans_module)
pca_kmeans_snapshot_module = importlib.reload(pca_kmeans_snapshot_module)
sweep_genes_snapshot_module = importlib.reload(sweep_genes_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
load_latest_metadata_snapshot = ncbi_metadata_snapshot_module.load_latest_metadata_snapshot
load_latest_sweep_genes_snapshot = (
    sweep_genes_snapshot_module.load_latest_sweep_genes_snapshot
)
resolve_pca_kmeans_snapshot = pca_kmeans_snapshot_module.resolve_pca_kmeans_snapshot
latest_pca_kmeans_snapshot_is_available = (
    pca_kmeans_snapshot_module.latest_pca_kmeans_snapshot_is_available
)

In [ ]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# CELL 3 — Define PCA/KMeans snapshot configuration
# =============================================================================

METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_metadata_csv"
)
SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "03-features" / "sweep_genes"
)
PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "04-analysis" / "pca_kmeans"
)

METADATA_SNAPSHOT_MODE = SnapshotMode.reuse_latest
SWEEP_GENES_SNAPSHOT_MODE = SnapshotMode.reuse_latest
PCA_KMEANS_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create

PCA_COMPONENT_COUNT_GRID = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 50, 100, 200)
KMEANS_CLUSTER_COUNT_GRID = tuple(range(2, 20))
PCA_SVD_SOLVER = "randomized"
PCA_RANDOM_STATE = 42
KMEANS_N_INIT = "auto"
SILHOUETTE_SAMPLE_SIZE = 8000
SILHOUETTE_RANDOM_STATE = 42
KMEANS_INITIALIZATION_REPEAT_COUNT = 6
SUBSAMPLE_REPEAT_COUNT = 6
SUBSAMPLE_FRACTION = 0.80
SUBSAMPLE_RANDOM_STATE = 123
MINIMUM_ACCEPTABLE_INIT_ARI_MIN = 0.70
MINIMUM_ACCEPTABLE_SUBSAMPLE_ARI_MIN = 0.60
EXPORT_PROJECTION_COMPONENT_COUNT = 3
UPDATE_LATEST_DIRECTORY = True

print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"SWeeP snapshot root directory: {SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA/KMeans output root directory: {PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA/KMeans snapshot mode: {PCA_KMEANS_SNAPSHOT_MODE}")
print(f"PCA component grid: {PCA_COMPONENT_COUNT_GRID}")
print(f"KMeans cluster grid: {KMEANS_CLUSTER_COUNT_GRID}")

In [ ]:
# =============================================================================
# CELL 4 — Resolve active source snapshots
# =============================================================================

metadata_snapshot_payload = load_latest_metadata_snapshot(
    snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
)
sweep_genes_snapshot_payload = load_latest_sweep_genes_snapshot(
    snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
)

metadata_snapshot_directory = metadata_snapshot_payload["snapshot_directory"]
metadata_manifest_file_path = metadata_snapshot_payload["manifest_file_path"]
metadata_csv_file_path = metadata_snapshot_payload["csv_file_path"]
metadata_manifest_payload = metadata_snapshot_payload["manifest"]

sweep_genes_snapshot_directory = sweep_genes_snapshot_payload["snapshot_directory"]
sweep_genes_manifest_file_path = sweep_genes_snapshot_payload["manifest_file_path"]
sweep_genes_manifest_payload = sweep_genes_snapshot_payload["manifest"]
sweep_genes_embeddings_file_path = sweep_genes_snapshot_payload["embeddings_file_path"]
sweep_genes_sequence_metadata_file_path = sweep_genes_snapshot_payload[
    "sequence_metadata_file_path"
]

print("Resolved source snapshots successfully.")
print(f"Metadata snapshot directory: {metadata_snapshot_directory}")
print(f"Metadata CSV path: {metadata_csv_file_path}")
print(f"SWeeP snapshot directory: {sweep_genes_snapshot_directory}")
print(f"SWeeP embeddings path: {sweep_genes_embeddings_file_path}")

In [ ]:
# =============================================================================
# CELL 5 — Resolve active PCA/KMeans snapshot
# =============================================================================

pca_kmeans_snapshot_payload = resolve_pca_kmeans_snapshot(
    snapshot_mode=PCA_KMEANS_SNAPSHOT_MODE,
    snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
    source_sweep_snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    pca_component_count_grid=PCA_COMPONENT_COUNT_GRID,
    kmeans_cluster_count_grid=KMEANS_CLUSTER_COUNT_GRID,
    pca_svd_solver=PCA_SVD_SOLVER,
    pca_random_state=PCA_RANDOM_STATE,
    kmeans_n_init=KMEANS_N_INIT,
    silhouette_sample_size=SILHOUETTE_SAMPLE_SIZE,
    silhouette_random_state=SILHOUETTE_RANDOM_STATE,
    kmeans_initialization_repeat_count=KMEANS_INITIALIZATION_REPEAT_COUNT,
    subsample_repeat_count=SUBSAMPLE_REPEAT_COUNT,
    subsample_fraction=SUBSAMPLE_FRACTION,
    subsample_random_state=SUBSAMPLE_RANDOM_STATE,
    minimum_acceptable_init_ari_min=MINIMUM_ACCEPTABLE_INIT_ARI_MIN,
    minimum_acceptable_subsample_ari_min=MINIMUM_ACCEPTABLE_SUBSAMPLE_ARI_MIN,
    export_projection_component_count=EXPORT_PROJECTION_COMPONENT_COUNT,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

pca_kmeans_snapshot_directory = pca_kmeans_snapshot_payload["snapshot_directory"]
pca_kmeans_manifest_file_path = pca_kmeans_snapshot_payload["manifest_file_path"]
pca_kmeans_manifest_payload = pca_kmeans_snapshot_payload["manifest"]
pca_coordinates_file_path = pca_kmeans_snapshot_payload["pca_coordinates_file_path"]
explained_variance_ratio_file_path = pca_kmeans_snapshot_payload[
    "explained_variance_ratio_file_path"
]
cluster_assignments_file_path = pca_kmeans_snapshot_payload[
    "cluster_assignments_file_path"
]
stability_grid_file_path = pca_kmeans_snapshot_payload["stability_grid_file_path"]
profiling_log_file_path = pca_kmeans_snapshot_payload["profiling_log_file_path"]
alignment_report_file_path = pca_kmeans_snapshot_payload["alignment_report_file_path"]
pca_coordinates = pca_kmeans_snapshot_payload["pca_coordinates"]
explained_variance_ratio = pca_kmeans_snapshot_payload["explained_variance_ratio"]
cluster_assignments_dataframe = pca_kmeans_snapshot_payload["cluster_assignments"]
stability_grid_dataframe = pca_kmeans_snapshot_payload["stability_grid"]
profiling_log_dataframe = pca_kmeans_snapshot_payload["profiling_log"]
alignment_report = pca_kmeans_snapshot_payload["alignment_report"]

print("Resolved PCA/KMeans snapshot successfully.")
print(f"Snapshot directory: {pca_kmeans_snapshot_directory}")
print(f"PCA coordinates path: {pca_coordinates_file_path}")
print(f"Cluster assignments path: {cluster_assignments_file_path}")
print(f"Stability grid path: {stability_grid_file_path}")

In [ ]:
# =============================================================================
# CELL 6 — Print PCA/KMeans snapshot summary
# =============================================================================

pca_kmeans_manifest_file_sha256 = sha256_of_file(
    input_file_path=pca_kmeans_manifest_file_path,
)
cluster_assignments_file_sha256 = sha256_of_file(
    input_file_path=cluster_assignments_file_path,
)
stability_grid_file_sha256 = sha256_of_file(
    input_file_path=stability_grid_file_path,
)
selected_configuration_summary_dataframe = pd.DataFrame(
    [
        {
            "selected_pca_component_count": pca_kmeans_manifest_payload[
                "selected_pca_component_count"
            ],
            "selected_cluster_count_k": pca_kmeans_manifest_payload[
                "selected_cluster_count_k"
            ],
            "selected_variance_explained_fraction": pca_kmeans_manifest_payload[
                "selected_variance_explained_fraction"
            ],
            "selected_silhouette_best_sampled": pca_kmeans_manifest_payload[
                "selected_silhouette_best_sampled"
            ],
            "selected_init_ari_min": pca_kmeans_manifest_payload[
                "selected_init_ari_min"
            ],
            "selected_subsample_ari_min": pca_kmeans_manifest_payload[
                "selected_subsample_ari_min"
            ],
            "selection_reason": pca_kmeans_manifest_payload["selection_reason"],
            "final_sampled_silhouette_value": pca_kmeans_manifest_payload[
                "final_sampled_silhouette_value"
            ],
        }
    ]
)

print("PCA/KMeans snapshot is ready.")
print(
    f"Snapshot created at UTC: {pca_kmeans_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Sequence count: {pca_kmeans_manifest_payload['sequence_count']}")
print(f"PCA coordinates shape: {pca_coordinates.shape}")
print(f"Explained variance vector length: {explained_variance_ratio.shape[0]}")
print(f"Cluster assignments rows: {len(cluster_assignments_dataframe)}")
print(f"Stability grid rows: {len(stability_grid_dataframe)}")
print(f"Alignment report: {alignment_report}")
print(f"Cluster assignments SHA-256: {cluster_assignments_file_sha256}")
print(f"Stability grid SHA-256: {stability_grid_file_sha256}")
print(f"Manifest SHA-256: {pca_kmeans_manifest_file_sha256}")

display(selected_configuration_summary_dataframe)

In [ ]:
# =============================================================================
# CELL 7 — Preview ranked configurations from the stability grid
# =============================================================================

stability_grid_ranked_dataframe = stability_grid_dataframe.sort_values(
    [
        "composite_score_silhouette_times_min_ari",
        "silhouette_best_sampled_filled",
        "variance_explained_fraction",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

print("Top ranked PCA/KMeans configurations:")
display(stability_grid_ranked_dataframe.head(10))

In [ ]:
# =============================================================================
# CELL 8 — Preview cluster assignments and PCA outputs
# =============================================================================

cluster_assignment_preview_row_limit = 10
cluster_assignment_preview_dataframe = cluster_assignments_dataframe.head(
    cluster_assignment_preview_row_limit
).copy()
cluster_size_summary_dataframe = (
    cluster_assignments_dataframe["cluster_label"]
    .value_counts()
    .rename_axis("cluster_label")
    .reset_index(name="row_count")
    .sort_values("cluster_label")
    .reset_index(drop=True)
)
pca_coordinate_preview_dataframe = pd.DataFrame(
    np.asarray(pca_coordinates[:5]),
    columns=[f"pc{i + 1}" for i in range(pca_coordinates.shape[1])],
)

print("Cluster assignments preview:")
display(cluster_assignment_preview_dataframe)
print("Cluster size summary:")
display(cluster_size_summary_dataframe)
print("Selected PCA coordinate preview:")
display(pca_coordinate_preview_dataframe.iloc[:, : min(10, pca_coordinate_preview_dataframe.shape[1])])

In [ ]:
# =============================================================================
# CELL 9 — Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- metadata_snapshot_directory")
print("- metadata_csv_file_path")
print("- sweep_genes_snapshot_directory")
print("- sweep_genes_embeddings_file_path")
print("- pca_kmeans_snapshot_directory")
print("- pca_coordinates_file_path")
print("- explained_variance_ratio_file_path")
print("- cluster_assignments_file_path")
print("- stability_grid_file_path")
print("- profiling_log_file_path")
print("- alignment_report_file_path")
print("- pca_kmeans_manifest_payload")
print("- pca_coordinates")
print("- explained_variance_ratio")
print("- cluster_assignments_dataframe")
print("- stability_grid_dataframe")
print("- stability_grid_ranked_dataframe")
print("- profiling_log_dataframe")
print("- alignment_report")
print("- selected_configuration_summary_dataframe")
print("- cluster_size_summary_dataframe")